In [ ]:
# CHECK DUPLICATES
 
import pandas as pd

# DataFrame
df = pd.read_csv("Data_infodoanhnghiep_clean.csv", encoding="utf-8-sig")

# Lấy tên cột theo index
cols = df.columns[[0, 2]]

# Lọc các dòng duplicate theo 2 cột đó
duplicates = df[df.duplicated(subset=cols, keep=False)]

duplicates.info()

In [ ]:
# XÓA DUPLICATES

import pandas as pd

# Đọc dữ liệu
df = pd.read_csv("Data_infodoanhnghiep_clean.csv", encoding="utf-8-sig")

# Xóa dòng trùng theo các cột index 0, 2 (giữ dòng đầu tiên)
df = df.drop_duplicates(subset=df.columns[[0, 2]], keep="first")

# Reset index (khuyến nghị)
df.reset_index(drop=True, inplace=True)

# Ghi đè lại vào file cũ
df.to_csv("Data_infodoanhnghiep_clean.csv", index=False, encoding="utf-8-sig")

# Kiểm tra nhanh
df.info()



In [ ]:
# CHUYỂN CSV SANG JSONL

import csv
import json

def csv_to_jsonl(csv_path, json_path):
    with open(csv_path, mode='r', encoding='utf-8-sig', newline='') as csv_file, \
         open(json_path, mode='w', encoding='utf-8') as json_file:

        reader = csv.DictReader(csv_file)
        for row in reader:
            json_file.write(
                json.dumps(row, ensure_ascii=False) + "\n"
            )

if __name__ == "__main__":
    csv_file_path = "Data_final_v2.csv"
    json_file_path = "Data_doanh_nghiep_v2.jsonl"
    csv_to_jsonl(csv_file_path, json_file_path)



In [ ]:
# ĐẾM SỐ PHẦN TỬ TRONG FILE JSON

import json

file_path = "Data_final.json" 

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Kiểm tra số phần tử
if isinstance(data, list):
    print(f"Số phần tử trong file JSON: {len(data)}")
elif isinstance(data, dict):
    print(f"Số phần tử (key) trong file JSON: {len(data)}")
else:
    print("Dữ liệu JSON không phải list hoặc dict")


In [ ]:
# SỬA LỖI ĐỊA CHỈ TRONG CSV

import pandas as pd
import re


VIET = r"A-Za-zÀ-Ỵà-ỵ"

ADDRESS_WORDS = [
    "Đường", "Phố", "Quốc lộ", "Ngõ", "Hẻm", "Tổ", "Khu",
    "Phường", "Quận", "Huyện", "Xã", "Thị trấn", "Thành phố"
]

ROAD_WORDS = [
    "Phố", "Đường", "Quốc lộ", "Ngõ", "Hẻm", "Khu", "Tổ"
]

def fix_address_final(s):
    if not isinstance(s, str):
        return s

    # 1. Tách chữ ↔ số
    s = re.sub(rf'([{VIET}])(\d+)', r'\1 \2', s)
    s = re.sub(rf'(\d+)([{VIET}])', r'\1 \2', s)

    # 2. Tách chữ cái đơn ↔ từ khóa địa chỉ (2 chiều)
    for w in ADDRESS_WORDS:
        s = re.sub(rf'([A-Z])({w})', r'\1 \2', s)
        s = re.sub(rf'({w})([A-Z])\b', r'\1 \2', s)

    # 3. Tách loại hình đường/phố ↔ tên riêng (PhốHuế → Phố Huế)
    for w in ROAD_WORDS:
        s = re.sub(rf'({w})([A-ZÀ-Ỵ])', r'\1 \2', s)

    # 4. Chuẩn hóa khoảng trắng
    s = re.sub(r'\s+', ' ', s)

    return s.strip()

df = pd.read_csv("Data_infodoanhnghiep_clean.csv", encoding="utf-8-sig")

df["Địa chỉ"] = df["Địa chỉ"].apply(fix_address_final)

df.to_csv("Data_infodoanhnghiep_fixed.csv", index=False, encoding="utf-8-sig")


In [ ]:
# KIỂM TRA KÝ TỰ LỖI

import pandas as pd

df = pd.read_csv("Data_infodoanhnghiep_fixed.csv", encoding="utf-8-sig")

# Lọc các dòng chứa ký tự lỗi �
mask_bad = df.astype(str).apply(
    lambda row: row.str.contains("�", regex=False).any(),
    axis=1
)

df_bad = df[mask_bad]

# Xem thử các dòng lỗi
print(df_bad)


In [ ]:
# SỬA KÝ TỰ LỖI
df = pd.read_csv("Data_infodoanhnghiep_fixed.csv", encoding="utf-8-sig")
CHAR_MAP = {
    "�": "",
    "â€™": "’",
    "â€“": "–",
    "â€”": "—",
    "â€œ": "“",
    "â€�": "”",
}

def normalize_text(s):
    if not isinstance(s, str):
        return s
    for k, v in CHAR_MAP.items():
        s = s.replace(k, v)
    return s

df = df.applymap(normalize_text)
df.to_csv("Data_infodoanhnghiep_fixed.csv", index=False, encoding="utf-8-sig")

In [ ]:
# HỢP NHẤT 2 DATAFRAME THEO MST

import pandas as pd
import re


df1 = pd.read_csv("Data_infodoanhnghiep_fixed.csv", encoding="utf-8-sig")

df2 = pd.read_csv("all_companies_data.csv", encoding="utf-8-sig")


# df1: giữ nguyên MST
df1["mst_key"] = df1["Mã số thuế"].astype(str).str.strip()

# df2: chỉ bỏ phần ngày cấp, GIỮ hậu tố
df2["mst_key"] = (
    df2["Mã số thuế"]
    .astype(str)
    .str.split(" - ").str[0]
    .str.strip()
)

merged = df1.merge(
    df2,
    how="outer",
    on="mst_key",
    suffixes=("_df1", "_df2")
)


# =========================
# 4. Chuẩn hóa schema đầu ra
# Ưu tiên df1, thiếu thì lấy df2
# =========================
final_df = pd.DataFrame({
    "Tên doanh nghiệp": merged["Tên doanh nghiệp"].combine_first(merged["Tên công ty"]),
    "Tên giao dịch": merged["Tên giao dịch"],
    "Mã số thuế": merged["Mã số thuế_df1"].combine_first(merged["mst_key"]),
    "Địa chỉ": merged["Địa chỉ_df1"].combine_first(merged["Địa chỉ_df2"]),
    "Tình trạng hoạt động": merged["Tình trạng hoạt động"].combine_first(merged["Trạng thái"]),
    "Ngày cấp giấy phép": merged["Ngày cấp giấy phép"],
    "Cơ quan thuế": merged["Cơ quan thuế"],
    "Phương pháp tính thuế": merged["Phương pháp tính thuế"],
    "Ngành nghề kinh doanh": merged["Ngành nghề kinh doanh"].combine_first(merged["Ngành nghề"]),
    "Chương khoản": merged["Chương khoản"],
})


# =========================
# 5. Kết quả
# =========================
print(f"Tổng số doanh nghiệp sau hợp nhất: {len(final_df)}")
print(final_df)
final_df.to_csv("Data_final_v2.csv", index=False, encoding="utf-8-sig")

In [ ]:
# SỬA LỖI ĐỊA CHỈ CÓ LOG VÀ NGÀNH NGHỀ KINH DOANH

import re


def clean_address(addr):
    if pd.isna(addr):
        return addr

    addr = str(addr)

    # Trường hợp có log hệ thống
    match = re.search(r"- Địa chỉ 1:\s*(.*?)\s*- Căn cứ:", addr)
    if match:
        return match.group(1).strip()

    # Trường hợp không có log → giữ nguyên
    return addr.strip()

final_df["Địa chỉ"] = final_df["Địa chỉ"].apply(clean_address)
final_df["Ngành nghề kinh doanh"] = (
    final_df["Ngành nghề kinh doanh"]
    .astype(str)
    .str.replace(r"\s*\(Xem danh sách\)", "", regex=True)
    .str.strip()
)

In [ ]:
#  HỢP NHẤT FILE JSON

import json

input_files = [
    "1900_reviews.json",
    "1900_reviews1-50.json",
    "1900_reviews101-200.json"
]

merged_data = []

for file in input_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        merged_data.extend(data)

with open("1900_merged.json", "w", encoding="utf-8") as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=2)


In [ ]:
# LỌC THÔNG TIN REVIEW TRONG JSONL

import json

INPUT_FILE = "company_with_reviews.jsonl"
OUTPUT_FILE = "company_with_reviews_clean.jsonl"

KEEP_REVIEW_FIELDS = {
    "rating",
    "role",
    "title",
    "meta",
    "pros",
    "cons"
}

total_reviews_before = 0
total_reviews_after = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:
        obj = json.loads(line)

        # Nếu có reviews thì lọc
        if "reviews" in obj and isinstance(obj["reviews"], list):
            cleaned_reviews = []

            for r in obj["reviews"]:
                total_reviews_before += 1

                cleaned = {
                    k: v for k, v in r.items()
                    if k in KEEP_REVIEW_FIELDS
                }

                # chỉ giữ review nếu còn nội dung
                if cleaned:
                    cleaned_reviews.append(cleaned)
                    total_reviews_after += 1

            if cleaned_reviews:
                obj["reviews"] = cleaned_reviews
            else:
                # nếu list review rỗng thì xóa key reviews
                obj.pop("reviews", None)

        # Ghi lại object (kể cả không có review)
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("DONE")
print("Reviews before:", total_reviews_before)
print("Reviews after :", total_reviews_after)
print("Output file  :", OUTPUT_FILE)


DONE
Reviews before: 47
Reviews after : 47
Output file  : company_with_reviews_clean.jsonl


In [ ]:
# LỌC THÔNG TIN REVIEW Ở ROOT JSONL

import json

INPUT_FILE = "MERGE9000\MERGE9000\merged.jsonl"
OUTPUT_FILE = "merged_clean.jsonl"

REVIEW_FIELDS = {
    "rating",
    "role",
    "title",
    "meta",
    "pros",
    "cons"
}

REMOVE_ROOT_FIELDS = {
    "company_name",
    "company_url",
    "rating",
    "role",
    "title",
    "meta",
    "pros",
    "cons",
    "_match_score",
    "_matched_company_name"
}

total_converted = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:
        obj = json.loads(line)

        # =========================
        # TÁCH REVIEW TỪ ROOT
        # =========================
        review = {
            k: obj.get(k)
            for k in REVIEW_FIELDS
            if k in obj
        }

        if review:
            obj["reviews"] = [review]
            total_converted += 1

        # =========================
        # XÓA FIELD THỪA Ở ROOT
        # =========================
        for k in REMOVE_ROOT_FIELDS:
            obj.pop(k, None)

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("DONE")
print("Converted records:", total_converted)
print("Output:", OUTPUT_FILE)


In [ ]:
# HỢP NHẤT REVIEWS TỪ 2 FILE JSONL

import json

BASE_FILE = "company_with_reviews_clean.jsonl"              # file chuẩn
EXTRA_FILE = "merged_clean.jsonl"
OUTPUT_FILE = "final_merged.jsonl"

KEY_FIELD = "Mã số thuế"

REVIEW_FIELD_ORDER = [
    "rating",
    "role",
    "title",
    "meta",
    "pros",
    "cons"
]

# =========================
# UTIL
# =========================
def normalize_review(r):
    return {k: r.get(k) for k in REVIEW_FIELD_ORDER if k in r}

def review_signature(r):
    return (r.get("title", ""), r.get("meta", ""))

# =========================
# LOAD EXTRA REVIEWS → MAP MST → LIST REVIEW
# =========================
review_map = {}

with open(EXTRA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        mst = obj.get(KEY_FIELD)
        reviews = obj.get("reviews", [])

        if mst and reviews:
            review_map.setdefault(mst, [])
            for r in reviews:
                review_map[mst].append(normalize_review(r))

# =========================
# PROCESS BASE FILE (KHÔNG XÓA DÒNG)
# =========================
total_lines = 0
lines_with_reviews = 0

with open(BASE_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:
        obj = json.loads(line)
        mst = obj.get(KEY_FIELD)

        base_reviews = obj.get("reviews", [])
        sigs = {review_signature(r) for r in base_reviews}

        extra_reviews = review_map.get(mst, [])

        for r in extra_reviews:
            sig = review_signature(r)
            if sig not in sigs:
                base_reviews.append(r)
                sigs.add(sig)

        if base_reviews:
            obj["reviews"] = base_reviews
            lines_with_reviews += 1

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
        total_lines += 1

# =========================
# REPORT
# =========================
print("DONE")
print("Tổng số dòng output:", total_lines)
print("Số dòng có reviews:", lines_with_reviews)


DONE
Tổng số dòng output: 1620401
Số dòng có reviews: 183


In [4]:
import json
import re
from collections import Counter

INPUT_FILE = "final_merged.jsonl"

word_counter = Counter()
total_docs = 0
total_words = 0
error_lines = 0

def tokenize(text: str):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            error_lines += 1
            continue   # bỏ qua dòng lỗi

        doc_text = json.dumps(obj, ensure_ascii=False)
        tokens = tokenize(doc_text)

        total_docs += 1
        total_words += len(tokens)
        word_counter.update(tokens)

# =========================
# METRICS
# =========================
avg_doc_length = total_words / total_docs if total_docs > 0 else 0
vocab_size = len(word_counter)

print("TỔNG SỐ DOCUMENT HỢP LỆ:", total_docs)
print("SỐ DÒNG LỖI JSON:", error_lines)
print("TỔNG SỐ TỪ:", total_words)
print("ĐỘ DÀI TRUNG BÌNH DOCUMENT:", round(avg_doc_length, 2))
print("VOCABULARY SIZE:", vocab_size)


TỔNG SỐ DOCUMENT HỢP LỆ: 1620401
SỐ DÒNG LỖI JSON: 0
TỔNG SỐ TỪ: 131985568
ĐỘ DÀI TRUNG BÌNH DOCUMENT: 81.45
VOCABULARY SIZE: 1688794
